# React — Props & component communication

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> This is the topic that removes the duplication you felt in the first mini-project.
>
> It also has a playground experiment: `playground/src/experiments/02-props.jsx`, used by
> all three lessons to watch props actually arrive and render.

## LESSON 14 — Passing and receiving props

In Mini-project 1 you wrote four KPI cards by hand because there was no way to write one
card and use it four times with different numbers. This is that way.

A parent passes values down as attributes:

**The React API:**

```jsx
<KpiCard label="Revenue" value="128,400 EUR" />
<KpiCard label="Orders"  value="1,284" />
```

Those values are called **props**. React collects them into **one object** and hands it to
your component as its single argument:

```jsx
function KpiCard(props) {
  return <p>{props.label}: {props.value}</p>;
}
```

Almost nobody writes it that way. The usual form pulls the pieces out in the signature:

```jsx
function KpiCard({ label, value }) {
  return <p>{label}: {value}</p>;
}
```

That is ordinary JavaScript destructuring, and React's docs say so plainly: it "is
equivalent to reading properties from a function parameter". Same object, shorter to read.

This also explains `children` from LESSON 5. There was never anything special about it —
`children` is simply one property of that same props object, the one React fills in with
whatever you nested between the tags.

### Props are read-only

React's word for this is **immutable**. A component may read its props and must never
change them:

> **Don't try to "change props".**

The reason is concrete, and there are two halves to it. React builds a fresh props object
for each render and treats it as yours to read, not to edit. And when a prop's **value** is
an object or an array — a card, a list, a user — that value is a reference the parent is
still holding. Change it from inside the child and you have quietly edited the parent's
data.

When a component needs different props, it does not edit them — the parent passes new ones.
How a parent decides to do that is topic 9.

### Key Notes

- A parent passes props as attributes; the component receives them as **one object**.
- `function Card({ label })` destructures that object — identical to reading `props.label`.
- `children` was always just one property of that same object.
- Props are **read-only**. Never assign to them.

### Example

**Runnable — plain JS.** Nothing here imitates React. A component's parameter *is* a plain
object, and destructuring it *is* plain destructuring — so this is the real mechanic, minus
the JSX.

Notice what happens to the repetition: four cards, one function.

In [ ]:
const l14kpis = [
  { label: "Revenue", value: "128,400 EUR" },
  { label: "Orders", value: "1,284" },
  { label: "Refunds", value: "37" },
  { label: "Active customers", value: "612" },
];

// The long form — read properties off the one argument.
function l14describe(props) {
  return `${props.label}: ${props.value}`;
}

// The usual form — destructure in the signature. Same object, same result.
function l14describeShort({ label, value }) {
  return `${label}: ${value}`;
}

for (const kpi of l14kpis) {
  console.log(l14describe(kpi), "|", l14describeShort(kpi));
}

### Exercise

Mini-project 1 also made you hand-write five activity rows. Same treatment.

**Part 1.** Write `l14rowText({ description, time })`, which returns
`"<description> — <time>"`. Destructure in the signature.

**Part 2.** Run it over this list and log each line:

```js
const l14activity = [
  { description: "Order #4821 shipped", time: "2 hours ago" },
  { description: "Invoice #1190 paid", time: "4 hours ago" },
  { description: "Refund issued for order #4790", time: "yesterday" },
  { description: "New customer: Baltic Foods", time: "yesterday", unread: true },
  { description: "Stock alert: Blue mugs below 20 units", time: "2 days ago" },
];
```

**Part 3.** The fourth object carries an extra property, `unread`, that your function never
mentions. Confirm it still works, then answer in a comment: why does the extra property
cause no trouble, and what would you have to do for the row to react to it?

**Part 4 — see it render.** **In the playground.** Open `playground/src/App.jsx`, point its one import line at
`./experiments/02-props.jsx`, and run `npm run dev`. The first three
`<KpiCard />` tags are one component used three times with different props — exactly the
thing you could not do in Mini-project 1. Change a `label` and watch only that card change.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

The function below breaks the read-only rule. Copy it into the cell, then work through the
steps.

```js
const l14card = { label: "Revenue", value: "128,400 EUR" };

function l14withCurrency(card) {
  card.label = card.label + " (EUR)";
  return `${card.label} — ${card.value}`;
}
```

1. Call it **twice with the same object** and log both results. They are not the same.
2. Log `l14card` afterwards. Whose object changed?
3. Rewrite it so it produces the same string on every call without touching the argument.
   Prove it by calling your version twice and logging the original object at the end.
4. One sentence: React calls props immutable. Given what you just saw, why does it matter
   that the parent is still holding the object it passed down?

In [ ]:
// Your code here

## LESSON 15 — What you can pass, and what comes back

React's docs are blunt about the range: you "can pass any JavaScript value through them,
including objects, arrays, and functions". Functions are the next lesson. Everything else
is this one — along with the four ways it goes wrong.

### Quotes give a string. Braces give the value.

**The React API:**

```jsx
<KpiCard label="Revenue" count={1284} />
```

`label` is the string `"Revenue"`. `count` is the **number** `1284`. This is the same rule
as LESSON 9, now applied to your own components — and the difference is not cosmetic:

```jsx
<KpiCard count="1284" />   {/* count + 1 is "12841" */}
<KpiCard count={1284} />   {/* count + 1 is 1285    */}
```

A number that arrives as a string is one of the quietest bugs in React, because it looks
right on screen until you do arithmetic with it.

### Booleans, and the shorthand

```jsx
<KpiCard featured={true} />
<KpiCard featured />        {/* identical — a bare prop name means true */}
```

Writing the name on its own is shorthand for `={true}`. There is no shorthand for false:
`featured={false}` has to be written out, and leaving the prop off entirely is the other
way to say "not featured".

Beware `featured="false"`. That is the **string** `"false"`, which is a non-empty string,
which is truthy. It will behave like `true`.

### Objects and arrays

```jsx
<Chart points={[12, 8, 30]} meta={{ unit: "EUR", live: true }} />
```

The double braces are not special syntax. The outer pair means "JavaScript goes here"; the
inner pair is an ordinary object literal — exactly the `style` situation from LESSON 9.

One reminder from LESSON 10: passing an object is fine, **rendering** one is not.
`{meta}` in your JSX throws *Objects are not valid as a React child*; `{meta.unit}` is what
you meant.

### Defaults

Give a prop a fallback in the destructuring:

```jsx
function KpiCard({ label, size = "md" }) { … }
```

React states the rule exactly, and it is worth reading twice:

> The default value is only used if the `size` prop is missing or if you pass
> `size={undefined}`. But if you pass `size={null}` or `size={0}`, the default value will
> **not** be used.

That is ordinary JavaScript destructuring behaviour, not a React invention. It catches
people out because "no value" and "the value null" feel like the same thing, and to a
default parameter they are not.

### Key Notes

- Quotes give a **string**; braces give the JavaScript **value**. `count={1284}` is a
  number, `count="1284"` is not.
- A bare prop name means `={true}`. `false` must be written out — and `"false"` is truthy.
- Objects and arrays go inside braces, which is why an object literal needs `{{ … }}`.
- A default fires only for a **missing** prop or `undefined` — never for `null`, `0`, `""`
  or `false`.

### Example

**Runnable — plain JS.** Defaults and value types are ordinary JavaScript, so the cell below
is the real behaviour rather than an imitation. Predict each line before you run it.

In [ ]:
function l15card({ label, count, size = "md", tone = "neutral" }) {
  return `${label} | count=${count} (${typeof count}) | size=${size} | tone=${tone}`;
}

// Braces give a number; quotes give a string.
console.log(l15card({ label: "Revenue", count: 1284 }));
console.log(l15card({ label: "Revenue", count: "1284" }));

// The default fires for a missing prop and for undefined...
console.log(l15card({ label: "A", count: 1 }));
console.log(l15card({ label: "B", count: 1, size: undefined }));

// ...and not for null, 0 or "".
console.log(l15card({ label: "C", count: 1, size: null }));
console.log(l15card({ label: "D", count: 1, size: 0 }));
console.log(l15card({ label: "E", count: 1, size: "" }));

### Exercise

Write `l15summary(props)` in the cell below. It takes one object and returns one line.

| prop | behaviour |
|---|---|
| `label` | always present, used as-is |
| `value` | if it is a **number**, format it with `.toLocaleString("en-US")`; if it is a string, use it unchanged |
| `unit` | defaults to `"EUR"` |
| `featured` | defaults to `false`; when true, the line starts with `"* "` |

The output format is `"<label>: <value> <unit>"`, with the `"* "` prefix when featured.

Call it with all five of these and log each result:

```js
{ label: "Revenue", value: 128400 }
{ label: "Revenue", value: "128,400" }
{ label: "Orders", value: 1284, unit: "items", featured: true }
{ label: "Refunds", value: 37, unit: null }
{ label: "Visitors", value: 612, unit: undefined }
```

Then answer in a comment: two of those five pass a `unit` that is not a real value — one
passes `null`, the other `undefined`. Only one of the two ends up with `"EUR"`. Which, and
why?

**Part 4 — see it render.** In `02-props.jsx` (playground), compare the two cards labelled
`value="1284"` and `value={1284}`: each prints what `typeof` says about the value it was
given. Then look at the **Refunds** card, which passes `unit={null}` — the unit is simply
missing from the line rather than falling back to `"EUR"`. That is this lesson's rule,
on screen.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

A colleague writes this and nothing works. There are **four** separate mistakes.

```jsx
<PriceTag amount="19.99" currency={EUR} discount=0 featured="false" />
```

For each one: what is wrong, and *when* do you find out — does the file fail to build, does
the browser throw, or does it run quietly and do the wrong thing?

Write your answers as comments. Then write the corrected tag.

Hint for the fourth: think about what `"false"` is, not what it says.

In [ ]:
// Your code here

## LESSON 16 — Functions as props: talking back to the parent

Data flows one way: a parent passes props down, and LESSON 14 established that the child
must never change them. So how does a child tell the parent that something happened?

The parent passes down a **function**, and the child calls it. Props can carry any
JavaScript value, and functions are on that list.

**The React API:**

```jsx
function ActivityRow({ label, id, onSelect }) {
  return <button onClick={() => onSelect(id)}>{label}</button>;
}

function Activity() {
  function handleSelect(id) {
    console.log("parent was told about", id);
  }

  return <ActivityRow label="Order #4821" id={4821} onSelect={handleSelect} />;
}
```

> The `onClick` above is topic 8's job and you are not expected to follow it yet. Ignore
> *how* the click is wired and look at *what travels*: a function goes down as `onSelect`,
> and the child calls it.

Read the direction of each arrow. `label` and `id` go **down**, from parent to child. The
call to `onSelect(id)` goes **up** — not as data the child writes into the parent, but as a
notification the parent chose to listen for.

### Pass the function. Do not call it.

This is the mistake everyone makes once, and React's docs name it flatly:

> Functions passed to event handlers must be passed, not called.

| correct | wrong |
|---|---|
| `onSelect={handleSelect}` | `onSelect={handleSelect()}` |

The reason is already familiar from LESSON 8: braces evaluate an expression **right away**.
`handleSelect()` is a call, so it runs during render, and what the child actually receives
is whatever the function returned — usually `undefined`.

### Naming

A convention, not a rule, and worth following because every React codebase uses it:

- the **prop** starts with `on` — `onSelect`, `onRemove`, `onSubmit`;
- the **function** starts with `handle` — `handleSelect`, `handleRemove`.

### What the parent can do with the news

Today: log it, count it, pass it to another function. What it *cannot* do yet is change
what is on screen — that needs state, and state is topic 9.

This is not a limitation of the pattern; it is the half of it you learn later. The wiring
in this lesson does not change when state arrives. Only the body of `handleSelect` does.

### Key Notes

- Data goes **down** as props; notifications come **back up** as function calls.
- `onSelect={handleSelect}` passes the function. `onSelect={handleSelect()}` calls it during
  render and passes the result.
- Convention: the prop is `onX`, the function is `handleX`.
- The **child** decides when to call it and what to pass; the **parent** decides what
  happens.

### Example

**Runnable — plain JS.** A function that takes another function and calls it later is
ordinary JavaScript — you did this in the JavaScript course. React adds no magic; it only
moves the callback along as a prop.

Notice the two separate moments: **building** the row, and the **later** moment the row is
acted on. In React the second moment is a click. Here it is just a call.

In [ ]:
// The parent's function. In React this would live in the parent component.
function l16handleSelect(id) {
  console.log("  parent was told about", id);
}

// The child. It receives the callback and decides when to use it.
function l16makeRow({ label, id, onSelect }) {
  return {
    label,
    onSelect,                   // kept so you can inspect what was handed over
    select: () => onSelect(id), // called later, not now
  };
}

const l16row = l16makeRow({
  label: "Order #4821",
  id: 4821,
  onSelect: l16handleSelect,
});

console.log("row built:", l16row.label); // nothing has been reported yet
l16row.select();                         // <- the moment of action
l16row.select();                         // the parent hears about it every time

### Exercise

**Part 1.** Using `l16makeRow` and `l16handleSelect` from the example cell, build a row for
each of these and put them in an array called `l16rows`:

```js
const l16activity = [
  { label: "Order #4821 shipped", id: 4821 },
  { label: "Invoice #1190 paid", id: 1190 },
  { label: "Refund for order #4790", id: 4790 },
];
```

Then act on the **first and third** rows only, and confirm from the output that the parent
was told about `4821` and `4790` — and never about `1190`.

**Part 2 — pass versus call.** Build two more rows: one with `onSelect: l16handleSelect`
and one with `onSelect: l16handleSelect(999)`. Before running, predict what each line
prints.

Then log `typeof` each row's **own** `onSelect` — `l16good.onSelect` and
`l16bad.onSelect` — and try calling `.select()` on both. One of them fails. Write down, in a
comment, the exact error and why the second row never had a usable callback.

**Part 3 — see it in React.** In `02-props.jsx` (playground), open the console and click
**Order #4821 shipped**: the parent's `handleSelect` reports the id. Now change
`onSelect={handleSelect}` to `onSelect={handleSelect(4821)}`, save, and reload — the console
fills before you touch anything, and the button then throws. Same two symptoms as the cell
above, in a real component.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

A colleague wants each row to report its own id and writes this:

```jsx
<ActivityRow label="Order #4821" onSelect={handleSelect(4821)} />
```

The parent's console fills up the moment the page loads, before anyone touches anything —
and clicking a row afterwards throws `TypeError: onSelect is not a function` instead of
reporting anything.

1. Explain both symptoms — the early output **and** the dead click — in terms of what
   `{ }` does with `handleSelect(4821)`.
2. `onSelect={handleSelect}` fixes the early output. But now the parent has no idea *which*
   row reported. Where should the `4821` come from instead, and which component is holding
   it already?
3. Write the corrected pair: the `<ActivityRow …>` tag as the parent writes it, and the one
   line inside `ActivityRow` that calls the callback.
4. In one sentence: why is it better for the **child** to supply the id than for the parent
   to bake it into the prop?

In [ ]:
// Your code here

## LESSON 17 — Slots: passing a component instead of adding a flag

LESSON 5 gave you `children`. React's docs describe what it really is:

> You can think of a component with a `children` prop as having a "hole" that can be
> "filled in" by its parent components with arbitrary JSX.

`children` is the **unnamed** hole. You can have named ones too, and it needs nothing new:
LESSON 7 established that an element is just a value, and LESSON 15 that a prop carries any
JavaScript value. Put the two together and a prop can hold JSX.

**The React API:**

```jsx
function Panel({ header, children }) {
  return (
    <section>
      <div>{header}</div>
      <div>{children}</div>
    </section>
  );
}
```

```jsx
<Panel header={<h2>Revenue</h2>}>
  <p>12,400 EUR</p>
</Panel>
```

Two holes: one filled by nesting, one filled by name. Leave `header` off entirely and it is
`undefined`, which renders nothing (LESSON 10) — no special handling required.

### Why this beats adding another flag

Here is the same panel built out of flags:

```jsx
<Panel title="Revenue" showTitle titleSize="lg" showIcon iconName="star" />
```

Every new variation costs two things: another prop, and another decision **inside** `Panel`
about what to do with it. Six screens later, `Panel` knows about every screen in the app,
and nobody can change it safely.

The slot version costs `Panel` nothing. It provides the shape; the caller provides the
content; `Panel` never learns what is in it. That is the same argument as LESSON 5's
`children`, now available wherever you need it rather than only in the nested position.

### Passing a component rather than an element

Sometimes the child needs to render the thing itself — perhaps more than once, perhaps
somewhere only it knows about. Then pass the **component**, not an element made from it:

```jsx
function Media({ Icon, children }) {
  return (
    <p>
      <Icon /> {children} <Icon />
    </p>
  );
}
```

```jsx
<Media Icon={StarIcon}>Featured</Media>
```

**The prop must be capitalised where you use it.** LESSON 4's rule looks at the first letter
of the tag and nothing else — it does not care that a lowercase name happens to be a
variable holding your component. Write `<icon />` and it compiles to `jsx("icon", …)`, an
HTML tag, and your component is never called.

Choosing between the two is simple: pass an **element** when the caller has already decided
exactly what it looks like, and a **component** when the child has to do the rendering.

### When a flag is still the right answer

Not every prop wants to be a slot. A flag is right when the variation genuinely belongs to
the component: `compact`, `disabled` — a small closed set, the component's own business, the
caller has no opinion about the markup.

The test is one question: **does this variation belong to the component, or to the caller?**
Unbounded variation belongs to the caller, and that is what a slot is for.

### Key Notes

- `children` is an unnamed hole; a named prop is a named one. Nothing new is required — a
  prop carries any value, and an element is a value.
- A slot the caller leaves out is `undefined`, and `undefined` renders nothing.
- A prop holding a **component** must be capitalised where it is used, or JSX reads it as an
  HTML tag and never calls it.
- Slot when the variation belongs to the caller; flag when it belongs to the component.

### Example

**The React API.** One `Panel`, two very different uses, and `Panel` knows about neither.

```jsx
<Panel header={<h2>Revenue</h2>}>
  <p>12,400 EUR</p>
</Panel>

<Panel header={<input placeholder="Search orders" />}>
  <p>No results yet.</p>
</Panel>
```

To support the second one with flags, `Panel` would have needed a `searchable` prop, a
`placeholder` prop, and a decision inside it about which header to build. With a slot it
needed nothing at all.

### Exercise

**In the playground.** There is no cell to run here — this lesson is about the shape of a
component's API, and shapes are judged by using them, not by printing them.

Point `playground/src/App.jsx` at `./experiments/02-props.jsx` and run `npm run dev`. The
file already contains a `Panel` and a `Media` for you to work against.

1. Use `Panel` **twice**: once passing a `header`, once leaving it off entirely. Confirm the
   second one renders without a header and without any error — you wrote no condition to
   make that happen.
2. Pass something other than a heading as the `header` — a `<button>`, an `<input>`,
   anything. Note that you did not touch `Panel` to do it.
3. Use `Media` with `Icon={StarIcon}`. It renders the icon twice; confirm both appear.
4. Now break it on purpose: inside `Media`, change one `<Icon />` to `<icon />`, lowercase.
   Save and look at the page and the console. What rendered in its place, and which LESSON 4
   message came back?
5. Put it back.

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

No code — four API decisions, written down.

For each, say **slot** or **flag**, and why in a few words:

1. `<Button loading />` — the button shows a spinner instead of its label while a request is
   in flight.
2. `<Card footer={…} />` — every screen's card footer is different: some have buttons, one
   has a timestamp, one has nothing.
3. `<Dialog title="Delete order?" />` — the title is always a short line of plain text.
4. `<Table emptyState={…} />` — what to show when there is nothing to list, which differs
   completely per table.

Then one more, and think before answering: what is the difference between

```jsx
<Media Icon={StarIcon} />
<Media icon={<StarIcon />} />
```

and when would you actually need the first? Number 3 is also worth a second look — not
everything that varies wants to be a slot.